# Canonical Elo Diagnostics by Map Position

This notebook isolates honest team-level Elo after canonical identity resolution and tests performance on Map 1 versus later maps.

# אבחון Elo קנוני לפי מיקום המפה בסדרה

מטרת הניסוי היא לבודד את אות ה־Elo האמיתי לאחר תיקון זהות הקבוצה. מזהי המקור המספריים מתחלפים בין משחקים ולכן הם עלולים לאפס את ההיסטוריה; כאן נעשה שימוש במפתחות הקנוניים בלבד. המודל מקבל דירוגים גלובליים ודירוגים לפי מפה ואת שני ההפרשים שלהם, ללא DNA, ללא H2H וללא פיצ'רים נוספים.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, brier_score_loss
from xgboost import XGBClassifier

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent
elo_module_dir = project_root / "src" / "features"
if str(elo_module_dir) not in sys.path:
    sys.path.insert(0, str(elo_module_dir))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from elo import PointInTimeEngine
from src.data.splitter import ChronologicalSplitter

RANDOM_SEED = 42

## Point-in-Time Features and Map Position

Ratings are captured before each update, while map position is derived within each series to expose possible intra-series momentum effects.

## בניית פיצ'רים נקודתיים בזמן ומיקום מפה

המנוע מחשב כל דירוג לפני שהוא מעדכן אותו בתוצאת המפה הנוכחית. לכל מפות הסדרה יש בדרך כלל אותו `datetime`, ולכן `game_id` העולה משמש כשובר שוויון בתוך `match_id`: השורה הראשונה היא מפה 1, וכל שורה נוספת היא מפה 2 ומעלה. משחקי Bo1 שייכים תמיד לקבוצת מפה 1.

החלוקה לפי מיקום נועדה לבדוק אם ביצועי המודל מגיעים בעיקר ממומנטום שנוצר לאחר תוצאת המפה הראשונה בסדרה. היא אינה מוסיפה את מיקום המפה עצמו למטריצת האימון.

In [2]:
source_columns = [
    "match_id", "game_id", "team1_id", "team1",
    "team2_id", "team2", "is_total", "bestOf",
    "score1_game", "score2_game", "map_name", "datetime",
    "team1_win", "team1_join_key", "team2_join_key",
]
source_path = project_root / "data" / "final_tournament_features.csv"
source_df = pd.read_csv(source_path, usecols=source_columns, low_memory=False)
source_df["source_order"] = np.arange(len(source_df))
elo_df = PointInTimeEngine(k_factor=24, initial_rating=1500).transform(source_df)
elo_df["map_position"] = elo_df.groupby("match_id", sort=False).cumcount() + 1

if elo_df[["team1_join_key", "team2_join_key"]].isna().any().any():
    raise ValueError("Canonical team keys must be complete.")
if not elo_df["datetime"].is_monotonic_increasing:
    raise AssertionError("Rows must remain chronologically ordered.")

splits = ChronologicalSplitter().split(elo_df)
train_base = splits["train"]
val_base = splits["val"]
locked_test_rows = len(splits["test"])

print(f"מספר שורות אימון: {len(train_base):,}")
print(f"מספר שורות אימות: {len(val_base):,}")
print(f"מספר שורות Test נעולות: {locked_test_rows:,}")
print(f"מפות 1 באימות: {val_base['map_position'].eq(1).sum():,}")
print(f"מפות 2 ומעלה באימות: {val_base['map_position'].gt(1).sum():,}")

מספר שורות אימון: 5,472
מספר שורות אימות: 633
מספר שורות Test נעולות: 595
מפות 1 באימות: 274
מפות 2 ומעלה באימות: 359


## Symmetrization and Elo-Only Matrix

Train and validation are mirrored independently. Only global Elo, map Elo, and their differences enter the model.

## סימטריזציה ובידוד מטריצת Elo

Train ו־Validation מסומטרים בנפרד לאחר החלוקה הכרונולוגית. בעותק המשוקף דירוגי שתי הקבוצות מוחלפים והתווית מתהפכת. הפרשי ה־Elo מחושבים רק לאחר ההחלפה, ולכן בדיקה מפורשת דורשת שהסימן שלהם יתהפך בדיוק. `map_position` נשמר ללא שינוי כדי ששני הכיוונים של אותה מפה יישארו באותה שכבת אבחון.

In [3]:
rating_pairs = [
    ("team1_elo_global", "team2_elo_global"),
    ("team1_elo_map", "team2_elo_map"),
]
identity_pairs = [
    ("team1_id", "team2_id"),
    ("team1", "team2"),
    ("team1_join_key", "team2_join_key"),
]
base_columns = [column for pair in rating_pairs for column in pair]
feature_columns = base_columns + ["elo_global_diff", "elo_map_diff"]
context_columns = [
    "match_id", "game_id", "datetime", "map_name", "map_position",
    "team1_id", "team1", "team2_id", "team2",
    "team1_join_key", "team2_join_key", "team1_win",
]

def symmetrize_elo(split_df):
    original = split_df[context_columns + base_columns].copy().reset_index(drop=True)
    mirrored = original.copy()
    for team1_column, team2_column in identity_pairs + rating_pairs:
        mirrored[team1_column] = original[team2_column].to_numpy(copy=True)
        mirrored[team2_column] = original[team1_column].to_numpy(copy=True)
    mirrored["team1_win"] = 1 - original["team1_win"].to_numpy()

    symmetric = pd.concat([original, mirrored], ignore_index=True)
    symmetric["elo_global_diff"] = (
        symmetric["team1_elo_global"] - symmetric["team2_elo_global"]
    )
    symmetric["elo_map_diff"] = (
        symmetric["team1_elo_map"] - symmetric["team2_elo_map"]
    )

    midpoint = len(original)
    for diff_column in ["elo_global_diff", "elo_map_diff"]:
        original_values = symmetric.iloc[:midpoint][diff_column].to_numpy()
        mirrored_values = symmetric.iloc[midpoint:][diff_column].to_numpy()
        if not np.allclose(original_values, -mirrored_values):
            raise AssertionError(f"Diff sign did not flip: {diff_column}")
    if not np.isclose(symmetric["team1_win"].mean(), 0.5):
        raise AssertionError("Symmetrization did not balance the target.")
    return symmetric

train_df = symmetrize_elo(train_base)
val_df = symmetrize_elo(val_base)
X_train = train_df[feature_columns]
y_train = train_df["team1_win"].astype(int)
X_val = val_df[feature_columns]
y_val = val_df["team1_win"].astype(int)

print(f"פיצ'רים במודל: {feature_columns}")
print(f"Train מסומטר: {len(train_df):,}")
print(f"Validation מסומטר: {len(val_df):,}")

פיצ'רים במודל: ['team1_elo_global', 'team2_elo_global', 'team1_elo_map', 'team2_elo_map', 'elo_global_diff', 'elo_map_diff']
Train מסומטר: 10,944
Validation מסומטר: 1,266


## Fixed Training Without Optuna

A fixed conservative configuration measures feature quality rather than search-budget effects, with early stopping used only for iteration selection.

## אימון קבוע ללא Optuna

כדי שהניסוי ימדוד את איכות הייצוג ולא את תקציב החיפוש, נעשה שימוש בתצורה קבועה ושמרנית. עצירה מוקדמת עוקבת אחר לוג־לוס ב־Validation ובוחרת את מספר העצים, אך לא מתבצע חיפוש היפר־פרמטרים.

In [4]:
canonical_elo_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.9,
    reg_lambda=2.0,
    tree_method="hist",
    random_state=RANDOM_SEED,
    n_jobs=-1,
    early_stopping_rounds=40,
)
canonical_elo_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=10,
)
print(f"האיטרציה הטובה ביותר: {canonical_elo_model.best_iteration}")

[0]	validation_0-logloss:0.69162	validation_1-logloss:0.69133


[10]	validation_0-logloss:0.68017	validation_1-logloss:0.67789


[20]	validation_0-logloss:0.67365	validation_1-logloss:0.67101


[30]	validation_0-logloss:0.66925	validation_1-logloss:0.66751

[40]	validation_0-logloss:0.66623	validation_1-logloss:0.66564


...

[90]	validation_0-logloss:0.65938	validation_1-logloss:0.66498

[100]	validation_0-logloss:0.65846	validation_1-logloss:0.66572


[110]	validation_0-logloss:0.65759	validation_1-logloss:0.66603


[115]	validation_0-logloss:0.65730	validation_1-logloss:0.66600

האיטרציה הטובה ביותר: 75


## Overall and Map-Stratified Metrics

Accuracy and Brier score are reported overall and separately for Map 1 and Map 2+, distinguishing pre-series forecasting from within-series prediction.

## מדדים כוללים ופילוח לפי מיקום המפה

הדיוק מודד את החלטת הסף ב־0.5, ואילו ציון ברייר מודד את איכות ההסתברות. המדדים מחושבים תחילה על כל Validation המסומטר, ולאחר מכן בנפרד למפה הראשונה ולמפות 2 ומעלה. מאחר שכל מפה מופיעה בשני כיוונים, הפילוח אינו מושפע מסדר הקבוצות המקורי.

In [5]:
val_probability = canonical_elo_model.predict_proba(X_val)[:, 1]
val_prediction = (val_probability >= 0.5).astype(int)

def calculate_metrics(mask):
    selected_target = y_val.loc[mask]
    selected_probability = val_probability[mask.to_numpy()]
    selected_prediction = val_prediction[mask.to_numpy()]
    return {
        "rows": int(mask.sum()),
        "accuracy": accuracy_score(selected_target, selected_prediction),
        "brier": brier_score_loss(selected_target, selected_probability),
    }

metric_groups = {
    "כלל האימות": pd.Series(True, index=val_df.index),
    "מפה 1": val_df["map_position"].eq(1),
    "מפה 2 ומעלה": val_df["map_position"].gt(1),
}
diagnostic_metrics = {
    group_name: calculate_metrics(mask) for group_name, mask in metric_groups.items()
}
for group_name, metrics in diagnostic_metrics.items():
    print(
        f"{group_name}: שורות={metrics['rows']:,}, "
        f"דיוק={metrics['accuracy']:.6f}, ברייר={metrics['brier']:.6f}"
    )

כלל האימות: שורות=1,266, דיוק=0.608215, ברייר=0.236055
מפה 1: שורות=548, דיוק=0.582117, ברייר=0.238658
מפה 2 ומעלה: שורות=718, דיוק=0.628134, ברייר=0.234068


## Step 1 Conclusion and Uncertainty Check

The observed map-position gap is descriptive until cluster bootstrap intervals determine whether it is distinguishable from sampling noise.

## מסקנת שלב 1 ומעבר לבדיקת אי־ודאות

המודל הקנוני השיג בכלל האימות דיוק של 0.608215 וציון ברייר של 0.236055. במפה הראשונה הדיוק הוא 0.582117 והברייר 0.238658, ואילו במפות 2 ומעלה הדיוק עולה ל־0.628134 והברייר משתפר ל־0.234068. כלומר, הדיוק לאחר המפה הראשונה גבוה ב־0.046017 והברייר נמוך ב־0.004590.

הפילוח תומך בהשערת המומנטום: חלק ניכר מהאות במפות המאוחרות נוצר לאחר שהמנוע כבר עדכן את Elo מתוצאה בתוך הסדרה. עבור סימולציה טרום־טורניר, שבה תוצאות הסדרה העתידית עדיין אינן ידועות, מדדי מפה 1 הם האומדן הישר והמחמיר יותר לאיכות דירוגי הפתיחה. השלב הבא בודק אם הפער יציב סטטיסטית.

# Step 1b — Match-Cluster Bootstrap Confidence Intervals

Entire matches, not individual rows, are resampled so correlated maps and mirrored copies remain together.

# שלב 1ב — רווח סמך באתחול מחדש ברמת משחק

מפות מאותה סדרה ושני העותקים הסימטריים שלהן אינם תצפיות עצמאיות. לכן יחידת הדגימה היא `match_id`: בכל אחת מאלף החזרות נדגמים משחקים שלמים עם החזרה, וכל השורות השייכות להם נכנסות יחד.

פער הדיוק מוגדר כדיוק מפות 2+ פחות דיוק מפה 1. פער הברייר מוגדר כברייר מפה 1 פחות ברייר מפות 2+, מפני שבברייר ערך נמוך יותר עדיף. בשני המדדים ערך חיובי מציין יתרון למפות המאוחרות. אם רווח הסמך כולל אפס, הנתונים אינם מספקים עדות יציבה לפער.

In [6]:
bootstrap_iterations = 1000
bootstrap_rng = np.random.default_rng(RANDOM_SEED)
is_map1 = val_df["map_position"].eq(1)
is_later_map = val_df["map_position"].gt(1)
correct = (val_prediction == y_val.to_numpy()).astype(float)
squared_error = (val_probability - y_val.to_numpy()) ** 2

bootstrap_rows = pd.DataFrame({
    "match_id": val_df["match_id"].to_numpy(),
    "map1_rows": is_map1.astype(int).to_numpy(),
    "map1_correct": correct * is_map1.to_numpy(),
    "map1_brier_sum": squared_error * is_map1.to_numpy(),
    "later_rows": is_later_map.astype(int).to_numpy(),
    "later_correct": correct * is_later_map.to_numpy(),
    "later_brier_sum": squared_error * is_later_map.to_numpy(),
})
cluster_metrics = bootstrap_rows.groupby("match_id", sort=False).sum()
cluster_values = cluster_metrics.to_numpy(dtype=float)
match_count = len(cluster_values)
sample_indices = bootstrap_rng.integers(
    0, match_count, size=(bootstrap_iterations, match_count)
)
sample_totals = cluster_values[sample_indices].sum(axis=1)
if (sample_totals[:, 0] == 0).any() or (sample_totals[:, 3] == 0).any():
    raise AssertionError("A bootstrap sample is missing one map-position stratum.")

map1_accuracy_samples = sample_totals[:, 1] / sample_totals[:, 0]
map1_brier_samples = sample_totals[:, 2] / sample_totals[:, 0]
later_accuracy_samples = sample_totals[:, 4] / sample_totals[:, 3]
later_brier_samples = sample_totals[:, 5] / sample_totals[:, 3]
accuracy_gap_samples = later_accuracy_samples - map1_accuracy_samples
brier_gap_samples = map1_brier_samples - later_brier_samples
accuracy_gap_ci = np.quantile(accuracy_gap_samples, [0.025, 0.975])
brier_gap_ci = np.quantile(brier_gap_samples, [0.025, 0.975])
point_accuracy_gap = (
    diagnostic_metrics["מפה 2 ומעלה"]["accuracy"]
    - diagnostic_metrics["מפה 1"]["accuracy"]
)
point_brier_gap = (
    diagnostic_metrics["מפה 1"]["brier"]
    - diagnostic_metrics["מפה 2 ומעלה"]["brier"]
)

print(f"מספר משחקים ייחודיים באימות: {match_count}")
print(f"מפה 1 — שורות גולמיות: {val_base['map_position'].eq(1).sum():,}; שורות מסומטרות: {is_map1.sum():,}")
print(f"מפה 2+ — שורות גולמיות: {val_base['map_position'].gt(1).sum():,}; שורות מסומטרות: {is_later_map.sum():,}")
print(
    f"פער דיוק נקודתי: {point_accuracy_gap:.6f}; "
    f"רווח סמך 95%: [{accuracy_gap_ci[0]:.6f}, {accuracy_gap_ci[1]:.6f}]"
)
print(
    f"פער ברייר נקודתי: {point_brier_gap:.6f}; "
    f"רווח סמך 95%: [{brier_gap_ci[0]:.6f}, {brier_gap_ci[1]:.6f}]"
)

מספר משחקים ייחודיים באימות: 274
מפה 1 — שורות גולמיות: 274; שורות מסומטרות: 548
מפה 2+ — שורות גולמיות: 359; שורות מסומטרות: 718
פער דיוק נקודתי: 0.046017; רווח סמך 95%: [-0.025326, 0.110082]
פער ברייר נקודתי: 0.004590; רווח סמך 95%: [-0.014511, 0.021938]


# Step 2 — Adding DNA to Canonical Elo

Pistol, CT-side, and T-side rates and their differences are added to test whether sparse round-level summaries improve canonical Elo.

# שלב 2 — הוספת DNA ל־Elo הקנוני

הניסוי השני מוסיף רק שלושה שיעורי DNA לכל צד: ניצחון בסיבובי פיסטול, ניצחון בצד CT וניצחון בצד T. לכל שיעור נוסף גם הפרש קבוצה 1 פחות קבוצה 2. ערכי DNA חסרים נשארים `NaN`, כדי ש־XGBoost ילמד עבורם מסלול חסרות במקום לקבל השלמה שרירותית.

טבלת ה־DNA מכילה מספר קטן של מפתחות שנבדלים רק באותיות גדולות וקטנות. לפני החיבור הם מאוחדים בממוצע משוקלל לפי מספר הסיבובים, ולאחר מכן נאכף חיבור רבים־לאחד לפי מפתח קבוצה קנוני ושם מפה. ה־DNA סטטי ואינו נקודתי בזמן; זו מגבלה ידועה של הניסוי.

In [7]:
dna_raw = pd.read_csv(project_root / "data" / "team_dna_features.csv")
dna_raw["dna_team_key"] = dna_raw["team"].astype("string").str.strip().str.casefold()
dna_rate_count_pairs = {
    "pistol_round_win_rate": "n_pistol_rounds",
    "ct_win_rate": "n_ct_rounds",
    "t_win_rate": "n_t_rounds",
}

def collapse_dna_group(group):
    collapsed = {}
    for rate_column, count_column in dna_rate_count_pairs.items():
        values = pd.to_numeric(group[rate_column], errors="coerce")
        weights = pd.to_numeric(group[count_column], errors="coerce").fillna(0)
        valid = values.notna() & weights.gt(0)
        collapsed[rate_column] = (
            np.average(values[valid], weights=weights[valid]) if valid.any() else np.nan
        )
    return pd.Series(collapsed)

dna = (
    dna_raw.groupby(["dna_team_key", "map_name"], sort=False)
    .apply(collapse_dna_group, include_groups=False)
    .reset_index()
)
if dna.duplicated(["dna_team_key", "map_name"]).any():
    raise AssertionError("DNA keys must be unique before joining.")

dna_rate_columns = list(dna_rate_count_pairs)

def merge_dna_rates(frame, side):
    lookup_key = f"team{side}_dna_lookup_key"
    side_dna = dna.rename(columns={
        "dna_team_key": lookup_key,
        **{column: f"team{side}_{column}" for column in dna_rate_columns},
    })
    before_rows = len(frame)
    merged = frame.merge(
        side_dna, how="left",
        left_on=[f"team{side}_join_key", "map_name"],
        right_on=[lookup_key, "map_name"],
        validate="many_to_one", sort=False,
    ).drop(columns=lookup_key)
    if len(merged) != before_rows:
        raise AssertionError(f"DNA join changed row count for side {side}.")
    return merged

train_dna_base = merge_dna_rates(merge_dna_rates(train_base, 1), 2)
val_dna_base = merge_dna_rates(merge_dna_rates(val_base, 1), 2)
print(f"כיסוי DNA בקבוצה 1 באימות: {val_dna_base[[f'team1_{c}' for c in dna_rate_columns]].notna().all(axis=1).mean():.1%}")
print(f"כיסוי DNA בקבוצה 2 באימות: {val_dna_base[[f'team2_{c}' for c in dna_rate_columns]].notna().all(axis=1).mean():.1%}")

כיסוי DNA בקבוצה 1 באימות: 93.5%
כיסוי DNA בקבוצה 2 באימות: 94.2%


## Elo + DNA Symmetrization and Matrix

Team-specific DNA columns are exchanged in mirrored rows and differences are recomputed after the exchange.

## סימטריזציה ומטריצת Elo + DNA

הסימטריזציה מתבצעת שוב ובנפרד על Train ועל Validation. דירוגי Elo ושיעורי ה־DNA מוחלפים בין הצדדים, התווית מתהפכת, וכל חמשת פיצ'רי ההפרש מחושבים לאחר ההחלפה. בדיקה אנטי־סימטרית כוללת גם ערכים חסרים ומבטיחה שכל הפרש בעותק המשוקף הוא הנגדי המדויק של המקור.

In [8]:
dna_pairs = [
    (f"team1_{column}", f"team2_{column}") for column in dna_rate_columns
]
dna_diff_pairs = {
    "dna_pistol_diff": ("team1_pistol_round_win_rate", "team2_pistol_round_win_rate"),
    "dna_ct_diff": ("team1_ct_win_rate", "team2_ct_win_rate"),
    "dna_t_diff": ("team1_t_win_rate", "team2_t_win_rate"),
}
all_diff_pairs = {
    "elo_global_diff": ("team1_elo_global", "team2_elo_global"),
    "elo_map_diff": ("team1_elo_map", "team2_elo_map"),
    **dna_diff_pairs,
}
dna_absolute_columns = [column for pair in dna_pairs for column in pair]
elo_dna_feature_columns = base_columns + dna_absolute_columns + list(all_diff_pairs)

def symmetrize_elo_dna(split_df):
    selected_columns = context_columns + base_columns + dna_absolute_columns
    original = split_df[selected_columns].copy().reset_index(drop=True)
    mirrored = original.copy()
    for team1_column, team2_column in identity_pairs + rating_pairs + dna_pairs:
        mirrored[team1_column] = original[team2_column].to_numpy(copy=True)
        mirrored[team2_column] = original[team1_column].to_numpy(copy=True)
    mirrored["team1_win"] = 1 - original["team1_win"].to_numpy()

    symmetric = pd.concat([original, mirrored], ignore_index=True)
    for diff_column, (team1_column, team2_column) in all_diff_pairs.items():
        symmetric[diff_column] = symmetric[team1_column] - symmetric[team2_column]

    midpoint = len(original)
    for diff_column in all_diff_pairs:
        original_values = symmetric.iloc[:midpoint][diff_column].to_numpy()
        mirrored_values = symmetric.iloc[midpoint:][diff_column].to_numpy()
        if not np.allclose(original_values, -mirrored_values, equal_nan=True):
            raise AssertionError(f"Diff sign did not flip: {diff_column}")
    if not np.isclose(symmetric["team1_win"].mean(), 0.5):
        raise AssertionError("Symmetrization did not balance the target.")
    return symmetric

train_dna = symmetrize_elo_dna(train_dna_base)
val_dna = symmetrize_elo_dna(val_dna_base)
X_train_dna = train_dna[elo_dna_feature_columns]
y_train_dna = train_dna["team1_win"].astype(int)
X_val_dna = val_dna[elo_dna_feature_columns]
y_val_dna = val_dna["team1_win"].astype(int)

print(f"מספר פיצ'רים במודל Elo + DNA: {len(elo_dna_feature_columns)}")
print(f"Train מסומטר: {len(train_dna):,}; Validation מסומטר: {len(val_dna):,}")

מספר פיצ'רים במודל Elo + DNA: 15
Train מסומטר: 10,944; Validation מסומטר: 1,266


## Fixed Training and Map-Stratified Evaluation

The same XGBoost configuration is retained so metric changes reflect DNA rather than hyperparameter tuning.

## אימון קבוע והערכת שכבות המפה

כדי לבודד את השפעת ה־DNA, נעשה שימוש באותה תצורת XGBoost קבועה של שלב 1 וללא Optuna. המדדים מחושבים על כלל Validation ועל אותן שתי שכבות מיקום, כך שהשינוי היחיד במטריצת הקלט הוא הוספת שיעורי ה־DNA והפרשיהם.

In [9]:
elo_dna_model = XGBClassifier(
    objective="binary:logistic", eval_metric="logloss",
    n_estimators=1000, learning_rate=0.03, max_depth=3,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.9,
    reg_lambda=2.0, tree_method="hist", random_state=RANDOM_SEED,
    n_jobs=-1, early_stopping_rounds=40,
)
elo_dna_model.fit(
    X_train_dna, y_train_dna,
    eval_set=[(X_train_dna, y_train_dna), (X_val_dna, y_val_dna)],
    verbose=10,
)
dna_probability = elo_dna_model.predict_proba(X_val_dna)[:, 1]
dna_prediction = (dna_probability >= 0.5).astype(int)

def calculate_dna_metrics(mask):
    selected_target = y_val_dna.loc[mask]
    selected_probability = dna_probability[mask.to_numpy()]
    selected_prediction = dna_prediction[mask.to_numpy()]
    return {
        "rows": int(mask.sum()),
        "accuracy": accuracy_score(selected_target, selected_prediction),
        "brier": brier_score_loss(selected_target, selected_probability),
    }

dna_metric_groups = {
    "כלל האימות": pd.Series(True, index=val_dna.index),
    "מפה 1": val_dna["map_position"].eq(1),
    "מפה 2 ומעלה": val_dna["map_position"].gt(1),
}
dna_diagnostic_metrics = {
    group_name: calculate_dna_metrics(mask)
    for group_name, mask in dna_metric_groups.items()
}
print(f"האיטרציה הטובה ביותר: {elo_dna_model.best_iteration}")
for group_name, metrics in dna_diagnostic_metrics.items():
    print(
        f"{group_name}: שורות={metrics['rows']:,}, "
        f"דיוק={metrics['accuracy']:.6f}, ברייר={metrics['brier']:.6f}"
    )

[0]	validation_0-logloss:0.69133	validation_1-logloss:0.69142


[10]	validation_0-logloss:0.67791	validation_1-logloss:0.67883


[20]	validation_0-logloss:0.66879	validation_1-logloss:0.67123


[30]	validation_0-logloss:0.66222	validation_1-logloss:0.66716


[40]	validation_0-logloss:0.65754	validation_1-logloss:0.66497


...

[70]	validation_0-logloss:0.64870	validation_1-logloss:0.66400

[80]	validation_0-logloss:0.64678	validation_1-logloss:0.66426

[90]	validation_0-logloss:0.64528	validation_1-logloss:0.66482


[100]	validation_0-logloss:0.64393	validation_1-logloss:0.66518

האיטרציה הטובה ביותר: 60
כלל האימות: שורות=1,266, דיוק=0.591627, ברייר=0.235670
מפה 1: שורות=548, דיוק=0.562044, ברייר=0.237011
מפה 2 ומעלה: שורות=718, דיוק=0.614206, ברייר=0.234647


## Step 2 Conclusion and Transition to H2H

DNA degrades the Elo signal, consistent with sparse noisy features attracting unhelpful tree splits. It is removed from the next experiment.

## מסקנת שלב 2 ומעבר לבידוד H2H

בשלב 1 פער הדיוק לטובת מפות 2+ הוא 0.046017, אך רווח הסמך ברמת משחק הוא ‎[-0.025326, 0.110082]‎. פער הברייר לטובת המפות המאוחרות הוא 0.004590, ורווח הסמך שלו הוא ‎[-0.014511, 0.021938]‎. שני הרווחים כוללים אפס: הדפוס מתאים להשערת המומנטום, אך אינו מספק לבדו עדות סטטיסטית יציבה במדגם של 274 משחקים.

מודל Elo + DNA השיג בכלל האימות דיוק של 0.591627 וברייר של 0.235670. במפה 1 התקבלו דיוק 0.562044 וברייר 0.237011; במפות 2+ התקבלו דיוק 0.614206 וברייר 0.234647. לעומת Elo בלבד, ה־DNA הוריד את הדיוק הכולל ב־0.016588 ושיפר את הברייר הכולל ב־0.000385 בלבד. במפה 1 הדיוק ירד ב־0.020073 והברייר השתפר ב־0.001647, ולכן אין כאן שדרוג משכנע לאות הטרום־סדרתי.

לאור התוצאה, ה־DNA מוסר מהניסוי הבא. שלב 3 חוזר ל־Elo הקנוני ומוסיף רק את היסטוריית המפגשים הישירים, ללא Optuna וללא שימוש ב־Test.

# Step 3 — Elo + Head-to-Head Isolation

Strictly prior matchup win counts are added without DNA to test their incremental value over canonical Elo.

# שלב 3 — בידוד Elo + מפגשים ישירים

מוני H2H מתארים כמה מפות ניצחה כל קבוצה מול היריבה לפני המשחק הנוכחי. המנוע דוחה את עדכון המונים עד שכל מפות אותו `match_id` קיבלו פיצ'רים, ולכן אף מפה אינה רואה תוצאה מתוך הסדרה שלה. המטריצה כוללת את שני המונים ואת ההפרש ביניהם לצד ששת פיצ'רי ה־Elo של שלב 1. DNA אינו נכלל.

הדגל `has_h2h` מציין שסכום הניצחונות ההיסטוריים גדול מאפס. הוא משמש לפילוח תוצאות בלבד ואינו נכנס למודל, כדי לבדוק אם מוני H2H עצמם מוסיפים אות כאשר קיימת היסטוריה.

## Symmetrizing Head-to-Head Counters

Team counters are exchanged under mirroring, the difference changes sign, and the coverage indicator remains unchanged.

## סימטריזציה של מוני המפגשים

בעותק המשוקף מוני הניצחונות של שתי הקבוצות מוחלפים, בעוד `has_h2h` נשאר ללא שינוי. הפרש H2H מחושב לאחר ההחלפה ונדרש להיות הנגדי המדויק של המקור. Train ו־Validation עוברים את הפעולה בנפרד לאחר החלוקה הכרונולוגית.

In [10]:
h2h_pair = ("team1_h2h_wins", "team2_h2h_wins")
h2h_diff_pairs = {
    "elo_global_diff": ("team1_elo_global", "team2_elo_global"),
    "elo_map_diff": ("team1_elo_map", "team2_elo_map"),
    "h2h_wins_diff": h2h_pair,
}
elo_h2h_feature_columns = (
    base_columns
    + ["elo_global_diff", "elo_map_diff"]
    + list(h2h_pair)
    + ["h2h_wins_diff"]
)

def symmetrize_elo_h2h(split_df):
    selected_columns = context_columns + base_columns + list(h2h_pair)
    original = split_df[selected_columns].copy().reset_index(drop=True)
    original["has_h2h"] = (
        original["team1_h2h_wins"] + original["team2_h2h_wins"]
    ).gt(0)
    mirrored = original.copy()
    for team1_column, team2_column in identity_pairs + rating_pairs + [h2h_pair]:
        mirrored[team1_column] = original[team2_column].to_numpy(copy=True)
        mirrored[team2_column] = original[team1_column].to_numpy(copy=True)
    mirrored["team1_win"] = 1 - original["team1_win"].to_numpy()

    symmetric = pd.concat([original, mirrored], ignore_index=True)
    for diff_column, (team1_column, team2_column) in h2h_diff_pairs.items():
        symmetric[diff_column] = symmetric[team1_column] - symmetric[team2_column]

    midpoint = len(original)
    for diff_column in h2h_diff_pairs:
        original_values = symmetric.iloc[:midpoint][diff_column].to_numpy()
        mirrored_values = symmetric.iloc[midpoint:][diff_column].to_numpy()
        if not np.allclose(original_values, -mirrored_values):
            raise AssertionError(f"Diff sign did not flip: {diff_column}")
    if not np.array_equal(
        symmetric.iloc[:midpoint]["has_h2h"].to_numpy(),
        symmetric.iloc[midpoint:]["has_h2h"].to_numpy(),
    ):
        raise AssertionError("H2H presence changed under mirroring.")
    if not np.isclose(symmetric["team1_win"].mean(), 0.5):
        raise AssertionError("Symmetrization did not balance the target.")
    return symmetric

train_h2h = symmetrize_elo_h2h(train_base)
val_h2h = symmetrize_elo_h2h(val_base)
X_train_h2h = train_h2h[elo_h2h_feature_columns]
y_train_h2h = train_h2h["team1_win"].astype(int)
X_val_h2h = val_h2h[elo_h2h_feature_columns]
y_val_h2h = val_h2h["team1_win"].astype(int)

print(f"מספר פיצ'רים במודל Elo + H2H: {len(elo_h2h_feature_columns)}")
print(f"שיעור שורות Validation עם H2H: {val_h2h['has_h2h'].mean():.1%}")
print(f"שורות עם H2H: {val_h2h['has_h2h'].sum():,}")
print(f"שורות ללא H2H: {(~val_h2h['has_h2h']).sum():,}")

מספר פיצ'רים במודל Elo + H2H: 9
שיעור שורות Validation עם H2H: 61.6%
שורות עם H2H: 780
שורות ללא H2H: 486


## Fixed Training and Three-Way Evaluation

Metrics are reported overall, by map position, and by whether previous head-to-head history exists.

## אימון קבוע והערכת שלוש זוויות

המודל משתמש באותה תצורה קבועה כמו שלבים 1 ו־2, ללא חיפוש פרמטרים. לאחר האימון נמדדים כלל Validation, מפה 1 מול מפות 2+, ושורות עם היסטוריית H2H מול שורות ללא היסטוריה. פילוח הכיסוי בוחן אם הפיצ'ר מועיל דווקא במקום שבו ערכו אינו אפס.

In [11]:
elo_h2h_model = XGBClassifier(
    objective="binary:logistic", eval_metric="logloss",
    n_estimators=1000, learning_rate=0.03, max_depth=3,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.9,
    reg_lambda=2.0, tree_method="hist", random_state=RANDOM_SEED,
    n_jobs=-1, early_stopping_rounds=40,
)
elo_h2h_model.fit(
    X_train_h2h, y_train_h2h,
    eval_set=[(X_train_h2h, y_train_h2h), (X_val_h2h, y_val_h2h)],
    verbose=10,
)
h2h_probability = elo_h2h_model.predict_proba(X_val_h2h)[:, 1]
h2h_prediction = (h2h_probability >= 0.5).astype(int)

def calculate_h2h_metrics(mask):
    selected_target = y_val_h2h.loc[mask]
    selected_probability = h2h_probability[mask.to_numpy()]
    selected_prediction = h2h_prediction[mask.to_numpy()]
    return {
        "rows": int(mask.sum()),
        "accuracy": accuracy_score(selected_target, selected_prediction),
        "brier": brier_score_loss(selected_target, selected_probability),
    }

h2h_metric_groups = {
    "כלל האימות": pd.Series(True, index=val_h2h.index),
    "מפה 1": val_h2h["map_position"].eq(1),
    "מפה 2 ומעלה": val_h2h["map_position"].gt(1),
    "קיימת היסטוריית H2H": val_h2h["has_h2h"],
    "לא קיימת היסטוריית H2H": ~val_h2h["has_h2h"],
}
h2h_diagnostic_metrics = {
    group_name: calculate_h2h_metrics(mask)
    for group_name, mask in h2h_metric_groups.items()
}
print(f"האיטרציה הטובה ביותר: {elo_h2h_model.best_iteration}")
for group_name, metrics in h2h_diagnostic_metrics.items():
    print(
        f"{group_name}: שורות={metrics['rows']:,}, "
        f"דיוק={metrics['accuracy']:.6f}, ברייר={metrics['brier']:.6f}"
    )

if not np.array_equal(val_df["match_id"].to_numpy(), val_h2h["match_id"].to_numpy()):
    raise AssertionError("Elo-only and Elo+H2H validation rows are not aligned.")
print("השוואה ישירה ל־Elo בלבד בתוך שכבות הכיסוי:")
for group_name in ["קיימת היסטוריית H2H", "לא קיימת היסטוריית H2H"]:
    mask = h2h_metric_groups[group_name]
    elo_only_metrics = calculate_metrics(mask)
    h2h_metrics = h2h_diagnostic_metrics[group_name]
    print(
        f"{group_name}: שינוי דיוק={h2h_metrics['accuracy'] - elo_only_metrics['accuracy']:+.6f}, "
        f"שיפור ברייר={elo_only_metrics['brier'] - h2h_metrics['brier']:+.6f}"
    )

[0]	validation_0-logloss:0.69162	validation_1-logloss:0.69133


[10]	validation_0-logloss:0.68085	validation_1-logloss:0.67871


[20]	validation_0-logloss:0.67361	validation_1-logloss:0.67073


[30]	validation_0-logloss:0.66914	validation_1-logloss:0.66721


[40]	validation_0-logloss:0.66608	validation_1-logloss:0.66555


...

[60]	validation_0-logloss:0.66244	validation_1-logloss:0.66519


[70]	validation_0-logloss:0.66110	validation_1-logloss:0.66513


[80]	validation_0-logloss:0.66007	validation_1-logloss:0.66556


[89]	validation_0-logloss:0.65915	validation_1-logloss:0.66573


האיטרציה הטובה ביותר: 49
כלל האימות: שורות=1,266, דיוק=0.609005, ברייר=0.236257
מפה 1: שורות=548, דיוק=0.585766, ברייר=0.239141
מפה 2 ומעלה: שורות=718, דיוק=0.626741, ברייר=0.234055
קיימת היסטוריית H2H: שורות=780, דיוק=0.624359, ברייר=0.232630
לא קיימת היסטוריית H2H: שורות=486, דיוק=0.584362, ברייר=0.242077
השוואה ישירה ל־Elo בלבד בתוך שכבות הכיסוי:
קיימת היסטוריית H2H: שינוי דיוק=+0.001282, שיפור ברייר=-0.000359
לא קיימת היסטוריית H2H: שינוי דיוק=+0.000000, שיפור ברייר=+0.000050


## Step 3 Conclusion and Transition to Rolling Form

The apparent benefit in covered matchups is attributed to selection bias: teams with H2H history also tend to have mature Elo ratings.

## מסקנת שלב 3 ומעבר ל־Rolling Form

מודל Elo + H2H השיג בכלל האימות דיוק של 0.609005 וברייר של 0.236257. במפה 1 התקבלו דיוק 0.585766 וברייר 0.239141; במפות 2+ התקבלו דיוק 0.626741 וברייר 0.234055. לעומת Elo-only, השינוי הכולל זעיר: הדיוק עלה ב־0.000790 והברייר הורע ב־0.000202.

בשורות עם היסטוריית H2H התקבלו דיוק 0.624359 וברייר 0.232630, לעומת דיוק 0.584362 וברייר 0.242077 ללא היסטוריה. עם זאת, ההשוואה הישירה ל־Elo-only על אותן שורות מראה שבקבוצת הכיסוי הדיוק עלה רק ב־0.001282 והברייר הורע ב־0.000359. לכן הביצועים הגבוהים יותר של קבוצת `has_h2h` משקפים בעיקר אוכלוסיית משחקים שונה, ולא תרומה שימושית של מוני H2H עצמם.

לאור התרומה הזניחה, H2H מוסר מהניסוי הבא. שלב 4 חוזר ל־Elo הקנוני ומוסיף רק מדדי כושר מתגלגלים שנבנו מהדאטה הראשי, ללא DNA, ללא H2H וללא Optuna.

# Step 4 — Elo + Five-Map Rolling Form

Pre-match win rate and round differential over each team's last five maps are tested as strictly chronological form indicators.

# שלב 4 — בידוד Elo + כושר חמש מפות אחרונות

לכל קבוצה מחושבים שני פיצ'רים לפני המשחק הנוכחי: שיעור הניצחונות בחמש המפות האחרונות וסכום הפרש הסיבובים באותן מפות. אם קיימות פחות מחמש מפות, החישוב משתמש בכל ההיסטוריה הזמינה; ללא היסטוריה מתקבל אפס.

כדי למנוע דליפת מומנטום מתוך סדרה, כל מפות אותו `match_id` מקבלות את תמונת הכושר שהייתה לפני תחילת המשחק. רק לאחר יצירת הפיצ'רים לכל הסדרה מתווספות תוצאות המפות לחלון המתגלגל. ניצחון במפה נקבע מתוצאת הסיבובים בפועל, והפרש הסיבובים נשמר מנקודת המבט של כל קבוצה.

## Rolling-Form Symmetrization and Matrix

Both teams' rolling features are exchanged in mirrored rows and their signed differences are recomputed.

## סימטריזציה ומטריצת Elo + כושר מתגלגל

שני פיצ'רי הכושר של קבוצה 1 מוחלפים עם מקביליהם בקבוצה 2 בעותק המשוקף. לאחר מכן מחושבים הפרש שיעור הניצחונות והפרש הפרש־הסיבובים, ונאכף היפוך סימן מלא. המטריצה כוללת רק את ששת פיצ'רי ה־Elo, ארבעת הפיצ'רים הגולמיים של הכושר ושני ההפרשים שלהם.

In [12]:
rolling_pairs = [
    ("team1_roll5_win_pct", "team2_roll5_win_pct"),
    ("team1_roll5_round_diff", "team2_roll5_round_diff"),
]
rolling_diff_pairs = {
    "elo_global_diff": ("team1_elo_global", "team2_elo_global"),
    "elo_map_diff": ("team1_elo_map", "team2_elo_map"),
    "roll5_win_pct_diff": ("team1_roll5_win_pct", "team2_roll5_win_pct"),
    "roll5_round_diff_diff": (
        "team1_roll5_round_diff", "team2_roll5_round_diff"
    ),
}
rolling_absolute_columns = [column for pair in rolling_pairs for column in pair]
elo_rolling_feature_columns = (
    base_columns + rolling_absolute_columns + list(rolling_diff_pairs)
)

def symmetrize_elo_rolling(split_df):
    selected_columns = context_columns + base_columns + rolling_absolute_columns
    original = split_df[selected_columns].copy().reset_index(drop=True)
    mirrored = original.copy()
    for team1_column, team2_column in identity_pairs + rating_pairs + rolling_pairs:
        mirrored[team1_column] = original[team2_column].to_numpy(copy=True)
        mirrored[team2_column] = original[team1_column].to_numpy(copy=True)
    mirrored["team1_win"] = 1 - original["team1_win"].to_numpy()

    symmetric = pd.concat([original, mirrored], ignore_index=True)
    for diff_column, (team1_column, team2_column) in rolling_diff_pairs.items():
        symmetric[diff_column] = symmetric[team1_column] - symmetric[team2_column]

    midpoint = len(original)
    for diff_column in rolling_diff_pairs:
        original_values = symmetric.iloc[:midpoint][diff_column].to_numpy()
        mirrored_values = symmetric.iloc[midpoint:][diff_column].to_numpy()
        if not np.allclose(original_values, -mirrored_values):
            raise AssertionError(f"Diff sign did not flip: {diff_column}")
    if not np.isclose(symmetric["team1_win"].mean(), 0.5):
        raise AssertionError("Symmetrization did not balance the target.")
    return symmetric

train_rolling = symmetrize_elo_rolling(train_base)
val_rolling = symmetrize_elo_rolling(val_base)
X_train_rolling = train_rolling[elo_rolling_feature_columns]
y_train_rolling = train_rolling["team1_win"].astype(int)
X_val_rolling = val_rolling[elo_rolling_feature_columns]
y_val_rolling = val_rolling["team1_win"].astype(int)

print(f"מספר פיצ'רים במודל Elo + Rolling Form: {len(elo_rolling_feature_columns)}")
print(f"Train מסומטר: {len(train_rolling):,}; Validation מסומטר: {len(val_rolling):,}")

מספר פיצ'רים במודל Elo + Rolling Form: 12
Train מסומטר: 10,944; Validation מסומטר: 1,266


## Fixed Training and Map-Position Evaluation

The unchanged model configuration isolates the contribution of rolling form across overall, Map 1, and Map 2+ strata.

## אימון קבוע ופילוח לפי מיקום מפה

המודל משתמש באותה תצורת XGBoost קבועה של כל מחקרי ההסרה, כדי שהשינוי במדדים ייוחס למטריצת הפיצ'רים ולא לחיפוש פרמטרים. לאחר האימון נמדדים כלל Validation, מפה 1 ומפות 2 ומעלה.

In [13]:
elo_rolling_model = XGBClassifier(
    objective="binary:logistic", eval_metric="logloss",
    n_estimators=1000, learning_rate=0.03, max_depth=3,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.9,
    reg_lambda=2.0, tree_method="hist", random_state=RANDOM_SEED,
    n_jobs=-1, early_stopping_rounds=40,
)
elo_rolling_model.fit(
    X_train_rolling, y_train_rolling,
    eval_set=[(X_train_rolling, y_train_rolling), (X_val_rolling, y_val_rolling)],
    verbose=10,
)
rolling_probability = elo_rolling_model.predict_proba(X_val_rolling)[:, 1]
rolling_prediction = (rolling_probability >= 0.5).astype(int)

def calculate_rolling_metrics(mask):
    selected_target = y_val_rolling.loc[mask]
    selected_probability = rolling_probability[mask.to_numpy()]
    selected_prediction = rolling_prediction[mask.to_numpy()]
    return {
        "rows": int(mask.sum()),
        "accuracy": accuracy_score(selected_target, selected_prediction),
        "brier": brier_score_loss(selected_target, selected_probability),
    }

rolling_metric_groups = {
    "כלל האימות": pd.Series(True, index=val_rolling.index),
    "מפה 1": val_rolling["map_position"].eq(1),
    "מפה 2 ומעלה": val_rolling["map_position"].gt(1),
}
rolling_diagnostic_metrics = {
    group_name: calculate_rolling_metrics(mask)
    for group_name, mask in rolling_metric_groups.items()
}
print(f"האיטרציה הטובה ביותר: {elo_rolling_model.best_iteration}")
for group_name, metrics in rolling_diagnostic_metrics.items():
    print(
        f"{group_name}: שורות={metrics['rows']:,}, "
        f"דיוק={metrics['accuracy']:.6f}, ברייר={metrics['brier']:.6f}"
    )

[0]	validation_0-logloss:0.69212	validation_1-logloss:0.69188


[10]	validation_0-logloss:0.68104	validation_1-logloss:0.67885


[20]	validation_0-logloss:0.67375	validation_1-logloss:0.67077


[30]	validation_0-logloss:0.66915	validation_1-logloss:0.66699


[40]	validation_0-logloss:0.66602	validation_1-logloss:0.66482


...

[90]	validation_0-logloss:0.65799	validation_1-logloss:0.66372


[100]	validation_0-logloss:0.65690	validation_1-logloss:0.66396


[110]	validation_0-logloss:0.65592	validation_1-logloss:0.66436


[113]	validation_0-logloss:0.65564	validation_1-logloss:0.66437

האיטרציה הטובה ביותר: 73
כלל האימות: שורות=1,266, דיוק=0.593207, ברייר=0.235635
מפה 1: שורות=548, דיוק=0.549270, ברייר=0.239329
מפה 2 ומעלה: שורות=718, דיוק=0.626741, ברייר=0.232816


## Final Ablation Decision

Rolling form also fails to improve the canonical Elo benchmark. DNA, H2H, and form are therefore excluded from the final production feature set.

## מסקנה ונקודת עצירה לאחר שלב 4

מודל Elo + Rolling Form השיג בכלל האימות דיוק של 0.593207 וברייר של 0.235635. במפה 1 התקבלו דיוק 0.549270 וברייר 0.239329; במפות 2+ התקבלו דיוק 0.626741 וברייר 0.232816.

לעומת Elo-only, הדיוק הכולל ירד ב־0.015008 והברייר השתפר ב־0.000420 בלבד. בשכבת מפה 1, שהיא האומדן הישיר למצב טרום־סדרה, הדיוק ירד ב־0.032847 והברייר הורע ב־0.000671. במפות 2+ הדיוק ירד ב־0.001393 והברייר השתפר ב־0.001252. לכן הכושר המתגלגל בתצורה זו אינו שדרוג משכנע לקו הבסיס הקנוני, ובפרט אינו משפר את חיזוי המפה הראשונה.

המחברת נעצרת לאחר בידוד Elo + Rolling Form. לא נעשה שימוש ב־DNA או H2H במודל זה, לא בוצע Optuna, ולא בוצעו סימטריזציה, חיזוי או הערכה על Test.